Setup and load the raw data

In [104]:
# Load packages
from pathlib import Path
import pandas as pd      # tables (DataFrames)
import numpy as np       # maths and conditions on columns

# Folders: raw files (read only) and cleaned files (output)
RAW = Path("../data/raw")
CLEAN = Path("../data/cleaned")


# Load the raw tables
sessions  = pd.read_csv(RAW / "website_sessions.csv")
pageviews = pd.read_csv(RAW / "website_pageviews.csv")
orders    = pd.read_csv(RAW / "orders.csv")
items     = pd.read_csv(RAW / "order_items.csv")
refunds   = pd.read_csv(RAW / "order_item_refunds.csv")
products  = pd.read_csv(RAW / "products.csv")


In [105]:
# Save the row counts now, so we can check later that cleaning didn't lose any rows
rows_before = {
    "sessions": len(sessions),
    "pageviews": len(pageviews),
    "orders": len(orders),
    "items": len(items),
    "refunds": len(refunds),
    "products": len(products),
}
rows_before
rows_before


{'sessions': 472871,
 'pageviews': 1188124,
 'orders': 32313,
 'items': 40025,
 'refunds': 1731,
 'products': 4}

1. Convert created_at to datetime

In [106]:
# Convert created_at from text to datetime in every table
for df in [sessions, pageviews, orders, items, refunds, products]:
    df["created_at"] = pd.to_datetime(df["created_at"])

# Check: created_at should now show datetime64 instead of str
orders.info()


<class 'pandas.DataFrame'>
RangeIndex: 32313 entries, 0 to 32312
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            32313 non-null  int64         
 1   created_at          32313 non-null  datetime64[us]
 2   website_session_id  32313 non-null  int64         
 3   user_id             32313 non-null  int64         
 4   primary_product_id  32313 non-null  int64         
 5   items_purchased     32313 non-null  int64         
 6   price_usd           32313 non-null  float64       
 7   cogs_usd            32313 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(5)
memory usage: 2.0 MB


2. Add traffic_channel column

Reason: missing UTM values are not errors. They mean the visit didn't come from a paid ad.
Keep the original columns unchanged and add one new column that labels every session.

In [107]:
# Decide the traffic channel for ONE session, based on its UTM source, campaign and referer
def get_channel(source, campaign, referer):
    if source in ["gsearch", "bsearch"] and campaign == "brand":
        return "paid_search_brand"        # paid ad, searched for the brand name
    if source in ["gsearch", "bsearch"]:
        return "paid_search_nonbrand"     # paid ad, general search (e.g. "teddy bear")
    if source == "socialbook":
        return "paid_social"              # paid ad on social media
    if pd.notna(referer):
        return "organic_search"           # no ad, but came from a search engine
    return "direct_type_in"               # no ad, no referer: typed the address / bookmark



In [108]:
# Run get_channel on every row: zip() takes the 3 columns side by side, one row at a time
sessions["traffic_channel"] = [
    get_channel(source, campaign, referer)
    for source, campaign, referer in zip(sessions["utm_source"],
                                         sessions["utm_campaign"],
                                         sessions["http_referer"])
]

# Check: how many sessions in each channel?
sessions["traffic_channel"].value_counts()


traffic_channel
paid_search_nonbrand    337615
organic_search           43411
paid_search_brand        41243
direct_type_in           39917
paid_social              10685
Name: count, dtype: int64

In [109]:
# 1. The new column should have no missing values (0 = good)
print("Missing traffic_channel:", sessions["traffic_channel"].isna().sum())

# 2. Compare with the original columns: each channel should match one clear pattern
pd.crosstab(sessions["traffic_channel"], sessions["utm_source"].fillna("missing"))


Missing traffic_channel: 0


utm_source,bsearch,gsearch,missing,socialbook
traffic_channel,,,,
direct_type_in,0,0,39917,0
organic_search,0,0,43411,0
paid_search_brand,7914,33329,0,0
paid_search_nonbrand,54909,282706,0,0
paid_social,0,0,0,10685


3. Flag the partial months

## Add is_partial_month (sessions, orders, order_items)
Reason: the data starts on 19 Mar 2012 and ends on 19 Mar 2015, so the first and last months are incomplete.
Flag them (1 = partial month) instead of deleting, so they can be left out of monthly trends but still counted in totals.



In [110]:
# The first and last month of the data (based on sessions, which covers the full period)
first_month = sessions["created_at"].min().to_period("M")
last_month  = sessions["created_at"].max().to_period("M")

print("First month:", first_month)
print("Last month:", last_month)


First month: 2012-03
Last month: 2015-03


In [111]:
# add the flag to the 3 tables 
# For each table: 1 if the row is in the first or last month, otherwise 0
for df in [sessions, orders, items]:
    month = df["created_at"].dt.to_period("M")                 # month of each row
    df["is_partial_month"] = month.isin([first_month, last_month]).astype(int)


In [112]:
# check

# Which months are flagged? Should only be 2012-03 and 2015-03
orders.loc[orders["is_partial_month"] == 1, "created_at"].dt.to_period("M").value_counts()


created_at
2015-03    1254
2012-03      60
Freq: M, Name: count, dtype: int64

In [113]:
# How many rows are flagged in each table?
for name, df in {"sessions": sessions, "orders": orders, "items": items}.items():
    print(name, "| partial-month rows:", df["is_partial_month"].sum(), "of", len(df))


sessions | partial-month rows: 16962 of 472871
orders | partial-month rows: 1314 of 32313
items | partial-month rows: 1753 of 40025


4. Add the profit columns

 Add profit_usd (orders, order_items)
Reason: profit per order and per product is the main measure for the business question.
Profit = price_usd − cogs_usd (gross profit, before refunds).


In [114]:
# Gross profit for each order and each item = price minus cost of goods sold
# round(2) keeps 2 decimals (cents), avoiding tiny computer rounding errors like 30.499999
orders["profit_usd"] = (orders["price_usd"] - orders["cogs_usd"]).round(2)
items["profit_usd"]  = (items["price_usd"] - items["cogs_usd"]).round(2)

# Check: look at a few rows
orders[["order_id", "price_usd", "cogs_usd", "profit_usd"]].head()


,order_id,price_usd,cogs_usd,profit_usd
0,1,49.99,19.49,30.5
1,2,49.99,19.49,30.5
2,3,49.99,19.49,30.5
3,4,49.99,19.49,30.5
4,5,49.99,19.49,30.5


In [115]:
# validate

# 1. No negative profits (0 = good); profiling showed cost is always below price
print("Negative profit orders:", (orders["profit_usd"] < 0).sum())

# 2. Order profit should equal the sum of its items' profit (0 = good)
item_profit = items.groupby("order_id")["profit_usd"].sum()
diff = (orders.set_index("order_id")["profit_usd"] - item_profit).abs()
print("Orders where profit ≠ sum of item profit:", (diff > 0.01).sum())


Negative profit orders: 0
Orders where profit ≠ sum of item profit: 0


In [116]:
# Profit per product (each product should have exactly one value, because prices are fixed)
items.groupby("product_id")["profit_usd"].unique()


product_id
1    [30.5]
2    [37.5]
3    [31.5]
4    [20.5]
Name: profit_usd, dtype: object

7. Add refund info to order_item

Reason: refunds are stored in a separate table. Adding them to each item lets us report profit before and after refunds.
New columns: is_refunded (1/0), refund_amount_usd, net_profit_usd = profit_usd − refund_amount_usd
Assumption: a refunded item is lost revenue (not resold) – the conservative choice.


In [117]:
# MErge:Left join: keep ALL items, add the refund amount where one exists
items = items.merge(
    refunds[["order_item_id", "refund_amount_usd"]],   # only the 2 columns we need from refunds
    on="order_item_id",                                 # the shared column used to match rows
    how="left",                                         # keep every item, even without a refund
)

# Check: row count must NOT change (still 40,025)
print("Rows before:", rows_before["items"], "| Rows after:", len(items))


Rows before: 40025 | Rows after: 40025


In [118]:
# create the refund columns

# 1 = item was refunded, 0 = not refunded (refund amount is filled only for refunded items)
items["is_refunded"] = items["refund_amount_usd"].notna().astype(int)

# Items without a refund: refund amount = 0 instead of empty
items["refund_amount_usd"] = items["refund_amount_usd"].fillna(0)

# Profit after refunds
items["net_profit_usd"] = (items["profit_usd"] - items["refund_amount_usd"]).round(2)

items[["order_item_id", "price_usd", "profit_usd", "is_refunded", "refund_amount_usd", "net_profit_usd"]].head()


,order_item_id,price_usd,profit_usd,is_refunded,refund_amount_usd,net_profit_usd
0,1,49.99,30.5,0,0.0,30.5
1,2,49.99,30.5,0,0.0,30.5
2,3,49.99,30.5,0,0.0,30.5
3,4,49.99,30.5,0,0.0,30.5
4,5,49.99,30.5,0,0.0,30.5


In [119]:
# validate

# 1. Number of refunded items should equal the number of refunds (1,731)
print("Refunded items:", items["is_refunded"].sum(), "| Refunds in table:", len(refunds))

# 2. Total refund amount should match the refunds table
print("Refund total in items:", items["refund_amount_usd"].sum().round(2))
print("Refund total in refunds:", refunds["refund_amount_usd"].sum().round(2))

# 3. A refunded item should have negative net profit (lost price > profit)
items.loc[items["is_refunded"] == 1, "net_profit_usd"].describe()


Refunded items: 1731 | Refunds in table: 1731
Refund total in items: 85338.69
Refund total in refunds: 85338.69


count    1731.000000
mean      -18.474402
std         2.772760
min       -22.490000
25%       -19.490000
50%       -19.490000
75%       -19.490000
max        -9.490000
Name: net_profit_usd, dtype: float64

**Why is net profit negative for refunded items?**
When an item is refunded, the shop returns the full price but has already paid the cost of the bear.
Example (Mr. Fuzzy): profit $30.50 − refund $49.99 = **−$19.49** → the shop loses the cost of the bear.
So a refund doesn't just remove the profit; it turns the sale into a loss.
*Assumption: refunded bears are not resold (conservative choice).*


6. Clean the products table

## Decisions 8 & 9 – Clean products
- Rename created_at → launched_at: it is the product's launch date, used for before/after comparisons
- Standardise "Mini bear" → "Mini Bear": consistent capital letters in all product names
- Add product_short_name: shorter labels for charts and Power BI


In [120]:
# Fix the name
# Remove any extra spaces at the start/end, then fix the lowercase "bear"
products["product_name"] = products["product_name"].str.strip()
products["product_name"] = products["product_name"].str.replace("Mini bear", "Mini Bear")


In [121]:
# add short names for the 3 product types (Mini Bear, Classic Bear, Giant Bear)
# Short name for each product ID (a dictionary: product_id → short name)
short_names = {
    1: "Mr. Fuzzy",
    2: "Love Bear",
    3: "Sugar Panda",
    4: "Mini Bear",
}

# .map() looks up each product_id in the dictionary and returns its short name
products["product_short_name"] = products["product_id"].map(short_names)


In [122]:
# Rename the date column
# Rename created_at to launched_at so its meaning is clear
products = products.rename(columns={"created_at": "launched_at"})


In [123]:
# Validate
# Only 4 rows, so look at the whole table
products


,product_id,launched_at,product_name,product_short_name
0,1,2012-03-19 08:00:00,The Original Mr. Fuzzy,Mr. Fuzzy
1,2,2013-01-06 13:00:00,The Forever Love Bear,Love Bear
2,3,2013-12-12 09:00:00,The Birthday Sugar Panda,Sugar Panda
3,4,2014-02-05 10:00:00,The Hudson River Mini Bear,Mini Bear


7. Validate the cleaned data
Validation – check that cleaning didn't break anything
Each `assert` checks one rule. If a rule is broken, the notebook stops with an error message.
If all pass, the cleaned data is safe to save.


In [124]:
# Row Counts
# 1. No rows lost or added during cleaning (compare with rows_before from Step 0)
rows_after = {
    "sessions": len(sessions),
    "pageviews": len(pageviews),
    "orders": len(orders),
    "items": len(items),
    "refunds": len(refunds),
    "products": len(products),
}
assert rows_after == rows_before, "Row counts changed during cleaning!"
print("Row counts unchanged:", rows_after)


Row counts unchanged: {'sessions': 472871, 'pageviews': 1188124, 'orders': 32313, 'items': 40025, 'refunds': 1731, 'products': 4}


In [125]:
# 2. Primary keys are still unique
assert sessions["website_session_id"].is_unique, "Duplicate session IDs!"
assert orders["order_id"].is_unique,             "Duplicate order IDs!"
assert items["order_item_id"].is_unique,         "Duplicate order item IDs (merge created copies?)"

# 3. Links between tables still work
assert orders["website_session_id"].isin(sessions["website_session_id"]).all(), "Orders without a session!"
assert items["order_id"].isin(orders["order_id"]).all(),                        "Items without an order!"
assert items["product_id"].isin(products["product_id"]).all(),                  "Items with unknown product!"

# 4. Dates are real dates now
for name, df in {"sessions": sessions, "pageviews": pageviews, "orders": orders,
                 "items": items, "refunds": refunds}.items():
    assert pd.api.types.is_datetime64_any_dtype(df["created_at"]), f"{name}: created_at is not a date!"
assert pd.api.types.is_datetime64_any_dtype(products["launched_at"]), "products: launched_at is not a date!"

# 5. New columns have no missing values
assert sessions["traffic_channel"].notna().all(),     "Missing traffic_channel!"
assert products["product_short_name"].notna().all(),  "Missing product short name!"
assert items["refund_amount_usd"].notna().all(),      "Missing refund amounts!"
assert items["net_profit_usd"].notna().all(),         "Missing net profit!"

# 6. Refunds: all 1,731 refunds made it into items
assert items["is_refunded"].sum() == len(refunds), "Refund count doesn't match!"

print("All checks passed ✅")


All checks passed ✅


8. Save the cleaned tables as csv


In [126]:
# All cleaned tables, with the name each file/table should get
cleaned = {
    "website_sessions": sessions,
    "website_pageviews": pageviews,
    "orders": orders,
    "order_items": items,
    "order_item_refunds": refunds,
    "products": products,
}

# Save each table as a CSV file (index=False = don't save pandas' row numbers as a column)
for name, df in cleaned.items():
    df.to_csv(CLEAN / f"{name}.csv", index=False)
    print("Saved", name, "→", len(df), "rows")


Saved website_sessions → 472871 rows
Saved website_pageviews → 1188124 rows
Saved orders → 32313 rows
Saved order_items → 40025 rows
Saved order_item_refunds → 1731 rows
Saved products → 4 rows


9. Load into PostgreSQL

In [127]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

# Load login details from .env (override=True = always use the latest values)
load_dotenv("../.env", override=True)

# Connect to PostgreSQL 
engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT"),
    database=os.getenv("PG_DATABASE"),
))

# test connection
with engine.connect() as conn:
    print("Connected to PostgreSQL!")


Connected to PostgreSQL!


In [128]:
# Load the table
# Write each cleaned table to the database (replace = overwrite if it exists)
for name, df in cleaned.items():
    df.to_sql(name, engine, if_exists="replace", index=False, chunksize=10_000)
    print("Loaded", name)


Loaded website_sessions
Loaded website_pageviews
Loaded orders
Loaded order_items
Loaded order_item_refunds
Loaded products


In [130]:
# Check row counts:
# Database row counts should match pandas
for name, df in cleaned.items():
    db_rows = pd.read_sql(f"SELECT COUNT(*) FROM {name}", engine).iloc[0, 0]
    print(name, len(df), db_rows, "✅" if db_rows == len(df) else "❌")



website_sessions 472871 472871 ✅
website_pageviews 1188124 1188124 ✅
orders 32313 32313 ✅
order_items 40025 40025 ✅
order_item_refunds 1731 1731 ✅
products 4 4 ✅
